# RAG Retrieval - BM25 Dataset Ingestion Notebook

This notebook demonstrates how to load transcript records from `tw3k_dataset.jsonl`, map them into `DocumentChunk` instances, build a `BM25Retriever` index, and test search queries.

## Step 1: Import Dependencies & Schema

In [ ]:
import json
from pathlib import Path
from src.schema import DocumentChunk
from src.bm25_retriever import BM25Retriever

## Step 2 & 3: Load `tw3k_dataset.jsonl` & Ingest as `DocumentChunk` Objects

In [ ]:
dataset_path = Path("tw3k_dataset.jsonl")
chunks = []

with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        chunk_id = data.get("chunk_id", "")
        content = data.get("text", "")
        metadata = {
            "video_id": data.get("video_id"),
            "video_title": data.get("video_title"),
            "formatted_time": data.get("formatted_time"),
            "timestamp_link": data.get("timestamp_link"),
            "channel": data.get("channel")
        }
        if content:
            chunks.append(DocumentChunk(id=chunk_id, content=content, metadata=metadata))

print(f"Successfully loaded {len(chunks)} DocumentChunk objects from {dataset_path.name}.")

## Step 4: Build BM25 Index

In [ ]:
print(f"Indexing {len(chunks)} chunks with BM25...")
bm25_retriever = BM25Retriever(chunks)
print("BM25 index successfully built!")

## Step 5: Execute Search Queries & Inspect BM25 Results

In [ ]:
queries = [
    "Overexplained tutorial armies generals units",
    "diplomacy coalition alliance vassal warlord",
    "public order corruption commandery tax revenue"
]

for query in queries:
    print(f"\nQUERY: '{query}'")
    print("=" * 60)
    results = bm25_retriever.search(query, top_k=3)
    for res in results:
        print(f"Rank {res.rank} | Score: {res.score:.4f} | ID: {res.chunk.id}")
        print(f"Video: {res.chunk.metadata.get('video_title')} ({res.chunk.metadata.get('formatted_time')})")
        print(f"Text snippet: {res.chunk.content[:150]}...")
        print("-" * 60)